In [ ]:
# ============================================================
# KAGGLE: WHOLE-BODY HEALTH SIMULATION + CAUSAL OPTIMIZATION
# Data: NHANES + BRFSS + NHIS (public downloads, no API keys)
# Engines embedded inline: Validation, Discovery, Attribution,
# Intervention, Optimization (based on your 5 notebooks)
# ============================================================

# ---------- 0) Environment ----------
import os, re, io, json, math, time, zipfile, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

# GPU / ML
import torch
import torch.nn as nn

# For discovery/intervention/optimization
import xgboost as xgb
import networkx as nx
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

# Optional (regime detection). If install fails, code will skip regimes.
try:
    import ruptures as rpt
except Exception:
    rpt = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

BASE_DIR = "/kaggle/working"
RAW_DIR = os.path.join(BASE_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

# ---------- 1) Robust download helpers ----------
def _http_get(url, timeout=60):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r

def download_to(url: str, out_path: str, overwrite: bool=False) -> str:
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if os.path.exists(out_path) and not overwrite:
        return out_path
    r = _http_get(url)
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path

def safe_read_xpt(path: str) -> pd.DataFrame:
    # NHANES commonly uses SAS XPT; pandas can read via read_sas.
    # (Works for many BRFSS releases too when SAS transport is provided.)
    df = pd.read_sas(path, format="xport", encoding="utf-8")
    # normalize columns
    df.columns = [str(c).strip() for c in df.columns]
    return df

# ---------- 2) NHANES: scrape XPT links & build merged table ----------
NHANES_CONTINUOUS_BASE = "https://wwwn.cdc.gov/nchs/nhanes/"

def nhanes_cycle_url(begin_year: int) -> str:
    # Example: BeginYear=2017 corresponds to 2017-2018
    return f"{NHANES_CONTINUOUS_BASE}continuousnhanes/default.aspx?BeginYear={begin_year}"

def nhanes_list_xpt_links(begin_year: int) -> List[str]:
    """
    Scrapes the NHANES cycle landing page and all sub-links it references
    to find .XPT URLs.
    """
    landing = nhanes_cycle_url(begin_year)
    html = _http_get(landing).text
    soup = BeautifulSoup(html, "html.parser")

    # Collect candidate links (relative or absolute)
    links = set()
    for a in soup.select("a[href]"):
        href = a.get("href", "")
        if not href:
            continue
        # make absolute
        if href.startswith("/"):
            href = "https://wwwn.cdc.gov" + href
        if href.startswith("http"):
            links.add(href)

    # Visit subpages that look like data pages and gather XPT links
    xpt_links = set()
    for u in list(links):
        if "datapage.aspx" in u or "search/datapage.aspx" in u or "continuousnhanes" in u:
            try:
                sub = _http_get(u).text
                s2 = BeautifulSoup(sub, "html.parser")
                for a in s2.select("a[href]"):
                    h = a.get("href", "")
                    if not h:
                        continue
                    if h.startswith("/"):
                        h = "https://wwwn.cdc.gov" + h
                    if h.lower().endswith(".xpt"):
                        xpt_links.add(h)
            except Exception:
                pass

    # Also capture any direct XPT links on landing itself
    for a in soup.select("a[href]"):
        h = a.get("href", "")
        if h.startswith("/"):
            h = "https://wwwn.cdc.gov" + h
        if h.lower().endswith(".xpt"):
            xpt_links.add(h)

    return sorted(xpt_links)

def nhanes_download_cycle(begin_year: int, max_files: int=40) -> List[str]:
    """
    Downloads up to max_files XPTs for the chosen cycle.
    Increase max_files if Kaggle disk allows, but NHANES can be huge.
    """
    xpts = nhanes_list_xpt_links(begin_year)
    if len(xpts) == 0:
        raise RuntimeError(f"No XPT links found for NHANES BeginYear={begin_year}. Try another cycle.")

    out_paths = []
    for i, url in enumerate(xpts[:max_files]):
        fn = url.split("/")[-1]
        out = os.path.join(RAW_DIR, f"nhanes_{begin_year}", fn)
        try:
            download_to(url, out)
            out_paths.append(out)
        except Exception:
            pass

    print(f"NHANES {begin_year}: downloaded {len(out_paths)}/{min(len(xpts), max_files)} XPT files")
    return out_paths

def nhanes_merge_on_seqn(xpt_paths: List[str], key: str="SEQN") -> pd.DataFrame:
    """
    Merges all tables that contain SEQN into one wide table.
    """
    tables = []
    for p in xpt_paths:
        try:
            df = safe_read_xpt(p)
            if key in df.columns:
                tables.append(df)
        except Exception:
            continue

    if not tables:
        raise RuntimeError("No NHANES tables with SEQN were readable.")

    # Start with the largest table that has SEQN
    tables = sorted(tables, key=lambda d: d.shape[1], reverse=True)
    base = tables[0].copy()
    for t in tables[1:]:
        # Avoid duplicate columns aside from key
        dup = set(base.columns).intersection(set(t.columns)) - {key}
        if dup:
            t = t.drop(columns=list(dup))
        base = base.merge(t, on=key, how="left")

    return base

# ---------- 3) BRFSS: parse annual page, download a SAS transport if present ----------
BRFSS_ANNUAL_PAGE = "https://www.cdc.gov/brfss/annual_data/annual_{year}.html"

def brfss_find_downloads(year: int) -> List[str]:
    url = BRFSS_ANNUAL_PAGE.format(year=year)
    html = _http_get(url).text
    soup = BeautifulSoup(html, "html.parser")
    zips = []
    for a in soup.select("a[href]"):
        h = a.get("href", "")
        if not h:
            continue
        if h.startswith("/"):
            h = "https://www.cdc.gov" + h
        if h.lower().endswith(".zip"):
            zips.append(h)
    return sorted(set(zips))

def brfss_download_year(year: int, prefer_keywords=("SAS", "XPT", "Transport", "Data")) -> List[str]:
    """
    Downloads ZIPs listed on the annual BRFSS year page.
    Then extracts locally. Returns extracted file paths.
    """
    links = brfss_find_downloads(year)
    if not links:
        print("No ZIP links found on BRFSS page for year:", year)
        return []

    out_dir = os.path.join(RAW_DIR, f"brfss_{year}")
    os.makedirs(out_dir, exist_ok=True)

    extracted = []
    for url in links:
        fn = url.split("/")[-1]
        zip_path = os.path.join(out_dir, fn)
        try:
            download_to(url, zip_path)
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(out_dir)
            for root, _, files in os.walk(out_dir):
                for f in files:
                    extracted.append(os.path.join(root, f))
        except Exception:
            continue

    print(f"BRFSS {year}: extracted {len(extracted)} files")
    return extracted

# ---------- 4) NHIS: (metadata-driven) placeholder loader ----------
# NHIS distribution formats vary; for breadth, treat NHIS as optional.
# Provide a "drop-in" hook: user can point to a public NHIS file URL and parse it.

def load_optional_nhis_csv(url: str) -> pd.DataFrame:
    """
    If you have a direct NHIS CSV download URL, this pulls it.
    (Some NHIS assets are not simple CSV; keep this as optional.)
    """
    r = _http_get(url)
    return pd.read_csv(io.StringIO(r.text))

# ============================================================
# 5) CLEANING LAYER (health-oriented)
# ============================================================

def coerce_numeric(df: pd.DataFrame, min_numeric_ratio=0.98) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == "object":
            # attempt numeric
            s = pd.to_numeric(out[c], errors="coerce")
            ratio = s.notna().mean()
            if ratio >= min_numeric_ratio:
                out[c] = s
    return out

def normalize_missing_codes(df: pd.DataFrame) -> pd.DataFrame:
    """
    NHANES/BRFSS often uses numeric sentinel codes (e.g., 7/9/77/99/7777/9999)
    depending on variable. We apply conservative heuristics:
    - if a column is integer-ish and has many 7/9 endings, null them out
    - if a column has very large repeated codes relative to quantiles, null them out
    """
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    for c in num_cols:
        s = out[c]
        # Common simple missing codes
        for code in [7, 9, 77, 99, 777, 999, 7777, 9999]:
            # only null if code is "rare-ish" relative to non-null
            frac = (s == code).mean()
            if 0.001 < frac < 0.25:
                out.loc[s == code, c] = np.nan

        # Extreme sentinel: if max is huge and appears often, null it
        if s.notna().any():
            vmax = s.max()
            if (s == vmax).mean() > 0.01 and vmax > s.quantile(0.99) * 1.5:
                out.loc[s == vmax, c] = np.nan
    return out

def basic_health_feature_pack(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates simple harmonized features if source columns exist.
    Keep this extensible; NHANES variable names differ by cycle/module.
    """
    out = df.copy()

    # Demographics (NHANES common)
    if "RIDAGEYR" in out.columns:
        out["AgeYears"] = out["RIDAGEYR"]
    if "RIAGENDR" in out.columns:
        out["Sex"] = out["RIAGENDR"]  # NHANES coding; interpret downstream
    if "BMXBMI" in out.columns:
        out["BMI"] = out["BMXBMI"]

    # Blood pressure (NHANES examples)
    # Use mean of available systolic/diastolic readings if present
    sys_cols = [c for c in out.columns if re.match(r"BPXSY\d+", str(c))]
    dia_cols = [c for c in out.columns if re.match(r"BPXDI\d+", str(c))]
    if sys_cols:
        out["SBP"] = out[sys_cols].mean(axis=1)
    if dia_cols:
        out["DBP"] = out[dia_cols].mean(axis=1)

    # Smoking status proxy (NHANES example)
    if "SMQ020" in out.columns:
        out["Smoked100Cigs"] = out["SMQ020"]

    # Diabetes proxy (NHANES example)
    if "DIQ010" in out.columns:
        out["DoctorToldDiabetes"] = out["DIQ010"]

    return out

# ============================================================
# 6) FRAMEWORK 1: TITAN VALIDATION FRAMEWORK (TVF) — from your notebook
#    (Trimmed to core; still multi-module)
# ============================================================

@dataclass
class TVFConfig:
    drift_alpha: float = 0.05
    reconstruction_error_threshold: float = 0.05
    min_explained_variance_ratio: float = 0.90
    max_null_spike: float = 0.10
    min_numeric_coercion: float = 0.98
    max_volumetric_drift: float = 0.50
    max_duplicate_rate: float = 0.0
    enable_benfords_law: bool = True
    max_correlation_drift: float = 0.25
    simpsons_threshold: float = 0.3
    max_fairness_disparity: float = 0.25
    max_cardinality: int = 100
    allow_zero_variance: bool = False
    max_leakage_r2: float = 0.98
    max_recency_days: int = 3650
    enable_iforest: bool = True
    iforest_contamination: float = 0.02

class TitanValidationFramework:
    def __init__(self, reference_data: pd.DataFrame, config: TVFConfig=None):
        self.config = config or TVFConfig()
        self.reference = reference_data.copy()
        self.num_cols = self.reference.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols = self.reference.select_dtypes(include=["object", "category"]).columns.tolist()
        self.ref_corr = self.reference[self.num_cols].corr() if self.num_cols else pd.DataFrame()
        self.ref_nulls = self.reference.isna().mean()
        self.ref_count = len(self.reference)
        self.ref_cat_freqs = {c: self.reference[c].value_counts(normalize=True) for c in self.cat_cols}
        self._train_unsupervised()
        self._train_structure()

    def _train_unsupervised(self):
        self.iforest = None
        if self.config.enable_iforest and len(self.reference) > 50 and self.num_cols:
            self.iforest = IsolationForest(
                contamination=self.config.iforest_contamination,
                random_state=42,
                n_jobs=-1
            )
            self.iforest.fit(self.reference[self.num_cols].fillna(0))

    def _train_structure(self):
        self.scaler = StandardScaler()
        if not self.num_cols:
            self.pca = None
            self.ref_recon_error = 0.0
            self.ref_explained_var = 1.0
            return
        X = self.reference[self.num_cols].fillna(0)
        Xs = self.scaler.fit_transform(X)
        from sklearn.decomposition import PCA
        self.pca = PCA(n_components=0.95)
        self.pca.fit(Xs)
        Xr = self.pca.inverse_transform(self.pca.transform(Xs))
        self.ref_recon_error = float(np.mean(np.square(Xs - Xr)))

    def validate(self, new_data: pd.DataFrame, target_col: str=None, date_col: str=None,
                 consistency_rules: List[str]=None, subgroups: List[str]=None) -> Tuple[bool, Dict]:
        report = {"modules": {}, "traffic_light": None}
        report["modules"]["ops"] = self._scan_ops(new_data)
        report["modules"]["drift_num"] = self._scan_drift_numeric(new_data)
        report["modules"]["structure"] = self._scan_structure(new_data)
        report["modules"]["leakage"] = self._scan_leakage(new_data, target_col)
        report["modules"]["logic"] = self._scan_logic(new_data, consistency_rules)
        report["traffic_light"] = self._generate_report(report)

        failed = (report["traffic_light"]["Status"] == "RED").any()
        return (not failed), report

    def _scan_ops(self, df):
        issues = []
        ratio = len(df) / (self.ref_count + 1e-9)
        if abs(1 - ratio) > self.config.max_volumetric_drift:
            issues.append(f"Volumetric Drift (Size Ratio: {ratio:.2f}x)")
        dupe_rate = df.duplicated().mean()
        if dupe_rate > self.config.max_duplicate_rate:
            issues.append(f"Duplicate Rows ({dupe_rate:.1%})")
        return {"issues": issues}

    def _scan_drift_numeric(self, df):
        if not self.num_cols:
            return {"drifted": []}
        drifted = []
        from scipy.stats import ks_2samp
        for c in self.num_cols:
            if c not in df.columns:
                continue
            a = self.reference[c].dropna()
            b = df[c].dropna()
            if len(a) < 50 or len(b) < 50:
                continue
            p = ks_2samp(a, b).pvalue
            if p < self.config.drift_alpha:
                drifted.append(c)
        return {"drifted": drifted}

    def _scan_structure(self, df):
        if not self.num_cols or self.pca is None:
            return {"issues": []}
        X = df[self.num_cols].fillna(0)
        Xs = self.scaler.transform(X)
        Xr = self.pca.inverse_transform(self.pca.transform(Xs))
        recon = float(np.mean(np.square(Xs - Xr)))
        ratio = recon / (self.ref_recon_error + 1e-9)
        issues = []
        if ratio > (1 + self.config.reconstruction_error_threshold):
            issues.append(f"Structure Shift (Error Ratio: {ratio:.2f}x)")
        return {"issues": issues, "recon_ratio": ratio}

    def _scan_logic(self, df, rules):
        issues = []
        if not rules:
            return {"issues": issues}
        for rule in rules:
            try:
                ok = df.eval(rule)
                if (~ok.fillna(True)).any():
                    issues.append(f"Rule Failed: {rule}")
            except Exception:
                issues.append(f"Rule Un-evaluable: {rule}")
        return {"issues": issues}

    def _scan_leakage(self, df, target):
        if not target or target not in df.columns:
            return {"issues": []}
        issues = []
        num = df.select_dtypes(include=[np.number]).copy()
        if target in num.columns:
            X = num.drop(columns=[target]).fillna(0)
            y = num[target].fillna(0)
            if X.shape[1] > 0 and len(X) > 200:
                m = LinearRegression().fit(X, y)
                r2 = m.score(X, y)
                if r2 > self.config.max_leakage_r2:
                    issues.append(f"Leakage Suspected (R2={r2:.3f})")
        return {"issues": issues}

    def _generate_report(self, report):
        rows = []
        for mod, out in report["modules"].items():
            if "issues" in out and out["issues"]:
                for iss in out["issues"]:
                    rows.append({"Module": mod, "Status": "RED", "Reason": iss})
            if "drifted" in out and out["drifted"]:
                rows.append({"Module": mod, "Status": "YELLOW", "Reason": f"Drifted cols: {len(out['drifted'])}"})
        if not rows:
            rows = [{"Module": "all", "Status": "GREEN", "Reason": "No critical issues detected"}]
        return pd.DataFrame(rows)

# ============================================================
# 7) FRAMEWORK 2: UNIFIED DISCOVERY ENGINE — adapted from your notebook
#    (health version: no "lags" required; focus on proxies + regimes)
# ============================================================

class UnifiedDiscoveryEngine:
    def __init__(self, df: pd.DataFrame, target_col: str, group_col: Optional[str]=None):
        self.raw_df = df.copy()
        self.target = target_col
        self.group_col = group_col
        self.known_features = [c for c in df.columns if c not in [target_col] + ([group_col] if group_col else [])]

    def scan_baseline(self):
        X = self.raw_df[self.known_features].fillna(0)
        y = self.raw_df[self.target].fillna(0)
        model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9)
        model.fit(X, y)
        r2 = model.score(X, y)
        return {"baseline_r2": float(r2)}

    def proxy_kill_dml(self, anchor_cols: List[str], suspect_cols: List[str], redundancy_r2=0.90, resid_corr=0.15):
        """
        Non-linear proxy check similar to your original: if a suspect is explainable by anchors
        and carries little independent residual signal vs target residual, drop it.
        """
        keep = set(anchor_cols)
        X_anchor = self.raw_df[anchor_cols].fillna(0)
        y_target = self.raw_df[self.target].fillna(0)

        # Residual of target on anchors
        m_t = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
        m_t.fit(X_anchor, y_target)
        resid_y = y_target - m_t.predict(X_anchor)

        for s in suspect_cols:
            if s not in self.raw_df.columns:
                continue
            y_s = self.raw_df[s].fillna(0).values.ravel()
            m_s = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
            m_s.fit(X_anchor, y_s)
            pred_s = m_s.predict(X_anchor)
            r2s = r2_score(y_s, pred_s)

            if r2s > redundancy_r2:
                resid_s = y_s - pred_s
                corr = np.corrcoef(resid_y, resid_s)[0,1]
                if np.isnan(corr) or abs(corr) < resid_corr:
                    # drop
                    if s in self.known_features:
                        self.known_features.remove(s)
                else:
                    keep.add(s)
            else:
                keep.add(s)

        self.known_features = [c for c in self.known_features if c in keep]
        return {"kept_features": self.known_features}

    def detect_regimes(self, split_col: str, min_group_n=500):
        """
        Finds whether "rules" differ by split_col (e.g., age decade, sex).
        """
        if split_col not in self.raw_df.columns:
            return {"status": "SKIP", "reason": "split col missing"}
        groups = self.raw_df[split_col].dropna().unique()
        if len(groups) < 2:
            return {"status": "SKIP", "reason": "not enough groups"}

        # Fit per-group linear model to compare coefficient drift
        feats = [c for c in self.known_features if c != split_col]
        drift = []
        for g in groups:
            sub = self.raw_df[self.raw_df[split_col] == g]
            if len(sub) < min_group_n:
                continue
            X = sub[feats].fillna(0)
            y = sub[self.target].fillna(0)
            m = Ridge(alpha=1.0).fit(X, y)
            drift.append((g, m.coef_))

        if len(drift) < 2:
            return {"status": "SKIP", "reason": "insufficient group fits"}

        # crude drift score = average pairwise cosine distance of coefficient vectors
        coefs = np.vstack([d[1] for d in drift])
        coefs = coefs / (np.linalg.norm(coefs, axis=1, keepdims=True) + 1e-9)
        sim = coefs @ coefs.T
        score = float(1 - sim[np.triu_indices_from(sim, k=1)].mean())
        status = "MULTI-REGIME" if score > 0.15 else "STABLE"
        return {"status": status, "coef_drift_score": score}

# ============================================================
# 8) FRAMEWORK 3: UNIVERSAL ATTRIBUTION VALIDATOR (IG + interactions) — GPU
# ============================================================

class UniversalAttributionValidator:
    def __init__(self, X: pd.DataFrame, y: pd.Series, hidden=64, lr=5e-3, epochs=300):
        self.features = list(X.columns)
        self.scaler = StandardScaler()
        Xs = self.scaler.fit_transform(X.values.astype(np.float32))
        self.X_tensor = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
        self.y_tensor = torch.tensor(y.values.astype(np.float32).reshape(-1,1), device=DEVICE)

        self.n_feat = self.X_tensor.shape[1]
        self.model = nn.Sequential(
            nn.Linear(self.n_feat, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, 1)
        ).to(DEVICE)

        opt = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-3)
        loss_fn = nn.MSELoss()

        self.model.train()
        for ep in range(epochs):
            opt.zero_grad()
            noise = torch.randn_like(self.X_tensor) * 0.03
            pred = self.model(self.X_tensor + noise)
            loss = loss_fn(pred, self.y_tensor)
            loss.backward()
            opt.step()

        self.model.eval()

    def integrated_gradients(self, steps=64, batch_size=2048) -> pd.Series:
        baseline = torch.zeros_like(self.X_tensor)
        attrs = []
        for i in range(0, len(self.X_tensor), batch_size):
            Xb = self.X_tensor[i:i+batch_size]
            Bb = baseline[i:i+batch_size]
            alphas = torch.linspace(0, 1, steps, device=DEVICE)
            path = Bb.unsqueeze(0) + alphas.view(-1,1,1) * (Xb - Bb).unsqueeze(0)
            path.requires_grad = True
            pred = self.model(path.reshape(-1, self.n_feat))
            grads = torch.autograd.grad(torch.sum(pred), path)[0]
            ig = (Xb - Bb) * torch.mean(grads, dim=0)
            attrs.append(ig.detach().cpu().numpy())
        ig_scores = pd.DataFrame(np.vstack(attrs), columns=self.features)
        self.ig_scores = ig_scores
        return ig_scores.mean().sort_values(ascending=False)

    def pair_interactions(self, top_pairs=10, sample_n=512) -> pd.Series:
        idx = np.random.choice(len(self.X_tensor), min(sample_n, len(self.X_tensor)), replace=False)
        Xs = self.X_tensor[idx].clone()
        base = torch.mean(self.X_tensor, dim=0)

        # prioritize correlated pairs (compute on CPU for convenience)
        Xraw = pd.DataFrame(self.scaler.inverse_transform(self.X_tensor.detach().cpu().numpy()), columns=self.features)
        corr = Xraw.corr().abs()
        pairs = [(a,b,corr.loc[a,b]) for a in self.features for b in self.features if a < b and corr.loc[a,b] > 0.3]
        pairs = sorted(pairs, key=lambda t: -t[2])[:200]  # cap compute

        inter = {}
        with torch.no_grad():
            for a,b,_ in pairs:
                ia = self.features.index(a)
                ib = self.features.index(b)
                X00 = Xs.clone(); X10 = Xs.clone(); X01 = Xs.clone(); X11 = Xs.clone()
                X00[:, [ia,ib]] = base[[ia,ib]]
                X10[:, ib] = base[ib]
                X01[:, ia] = base[ia]
                val = (self.model(X11) - self.model(X10) - self.model(X01) + self.model(X00)).mean().item()
                inter[f"{a} + {b}"] = val
        s = pd.Series(inter).sort_values(key=lambda z: np.abs(z), ascending=False).head(top_pairs)
        return s

# ============================================================
# 9) FRAMEWORK 4: UNIFIED INTERVENTION (PlatinumCausalEngine) — GPU XGB
# ============================================================

class PlatinumCausalEngine:
    def __init__(self, causal_graph, data: pd.DataFrame, verbose=True):
        self.G = self._build_graph(causal_graph)
        self.df = data.copy()
        self.verbose = verbose
        self.nodes = list(nx.topological_sort(self.G))
        self.models = {}
        self.model_types = {}
        self.residuals = pd.DataFrame(index=self.df.index)

        if self.verbose:
            print("Fitting structural models...")
        self._fit_adaptive_models()
        self._compute_residuals()

    def _build_graph(self, g_input):
        if isinstance(g_input, list):
            G = nx.DiGraph()
            G.add_edges_from(g_input)
            return G
        return g_input.copy()

    def _fit_adaptive_models(self):
        for node in self.nodes:
            parents = list(self.G.predecessors(node))
            if not parents:
                self.residuals[node] = self.df[node]
                continue

            X = self.df[parents].fillna(0)
            y = self.df[node].fillna(0)

            lin_scores, xgb_scores = [], []
            kf = KFold(n_splits=3, shuffle=True, random_state=42)

            for tr, va in kf.split(X):
                Xtr, Xva = X.iloc[tr], X.iloc[va]
                ytr, yva = y.iloc[tr], y.iloc[va]

                lin = LinearRegression().fit(Xtr, ytr)
                lin_scores.append(lin.score(Xva, yva))

                xg = xgb.XGBRegressor(
                    n_estimators=120, max_depth=4, learning_rate=0.05,
                    subsample=0.9, colsample_bytree=0.9,
                    device=("cuda" if torch.cuda.is_available() else "cpu")
                )
                xg.fit(Xtr, ytr)
                xgb_scores.append(xg.score(Xva, yva))

            avg_lin, avg_xgb = float(np.mean(lin_scores)), float(np.mean(xgb_scores))

            if avg_xgb > avg_lin + 0.05:
                m = xgb.XGBRegressor(
                    n_estimators=300, max_depth=5, learning_rate=0.03,
                    subsample=0.9, colsample_bytree=0.9,
                    device=("cuda" if torch.cuda.is_available() else "cpu")
                )
                m.fit(X, y)
                self.models[node] = m
                self.model_types[node] = "XGBoost"
            else:
                m = LinearRegression().fit(X, y)
                self.models[node] = m
                self.model_types[node] = "Linear"

            if self.verbose:
                print(f" - {node}: {self.model_types[node]} (cvR2={max(avg_lin, avg_xgb):.3f})")

    def _compute_residuals(self):
        for node in self.nodes:
            parents = list(self.G.predecessors(node))
            if not parents:
                continue
            X = self.df[parents].fillna(0)
            pred = self.models[node].predict(X)
            self.residuals[node] = self.df[node].fillna(0) - pred

    def _run_simulation_pass(self, df_base, res_base, treatment, target):
        df_sim = df_base.copy()
        for v, val in treatment.items():
            df_sim[v] = val

        for node in self.nodes:
            if node in treatment:
                continue
            parents = list(self.G.predecessors(node))
            if not parents:
                continue
            X = df_sim[parents].fillna(0)
            base_val = self.models[node].predict(X)
            df_sim[node] = base_val + res_base[node].values

        return float(df_sim[target].mean())

    def simulate_intervention(self, treatment: dict, target: str, n_boot=50):
        mu = self._run_simulation_pass(self.df, self.residuals, treatment, target)
        if n_boot <= 0:
            return {"E_y_do": mu, "ci_lower": mu, "ci_upper": mu, "std_error": 0.0}

        est = []
        for _ in range(n_boot):
            res_boot = self.residuals.sample(frac=1.0, replace=True, random_state=None)
            df_boot = self.df.loc[res_boot.index].copy()
            est.append(self._run_simulation_pass(df_boot, res_boot, treatment, target))
        est = np.array(est)
        return {
            "E_y_do": mu,
            "std_error": float(est.std()),
            "ci_lower": float(np.percentile(est, 2.5)),
            "ci_upper": float(np.percentile(est, 97.5)),
        }

# ============================================================
# 10) FRAMEWORK 5: UNIFIED OPTIMIZER — from your notebook (core)
# ============================================================

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, Matern
from scipy.optimize import minimize, differential_evolution
from scipy.stats import qmc, norm

class UnifiedOptimizer:
    def __init__(self, objective_fn, bounds: Dict[str, Tuple[float,float]]):
        self.objective_fn = objective_fn
        self.bounds = bounds
        self.param_names = list(bounds.keys())
        self.history = {"params": [], "values": [], "method": []}

    def _dict_to_array(self, d): return np.array([d[k] for k in self.param_names], dtype=float)
    def _array_to_dict(self, a): return dict(zip(self.param_names, list(map(float, a))))

    def _evaluate(self, d, method):
        v = float(self.objective_fn(**d))
        self.history["params"].append(d)
        self.history["values"].append(v)
        self.history["method"].append(method)
        return v

    def bayesian_optimize(self, n_init=8, n_iter=25, maximize=False):
        X, y = [], []
        sampler = qmc.LatinHypercube(d=len(self.param_names))
        init = sampler.random(n=n_init)

        for s in init:
            d = {}
            for j,(name,(lo,hi)) in enumerate(self.bounds.items()):
                d[name] = float(lo + s[j]*(hi-lo))
            val = self._evaluate(d, "bayesian")
            X.append(self._dict_to_array(d)); y.append(val)

        best = np.max(y) if maximize else np.min(y)

        for _ in range(n_iter):
            kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
            gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=5)
            gp.fit(np.array(X), np.array(y))

            best_candidate, best_ei = None, -np.inf
            cand = qmc.LatinHypercube(d=len(self.param_names)).random(n=600)

            for s in cand:
                d = {}
                for j,(name,(lo,hi)) in enumerate(self.bounds.items()):
                    d[name] = float(lo + s[j]*(hi-lo))
                xa = self._dict_to_array(d).reshape(1,-1)
                mu, sigma = gp.predict(xa, return_std=True)
                mu, sigma = float(mu), float(sigma)

                if maximize:
                    imp = mu - best
                else:
                    imp = best - mu
                Z = imp / (sigma + 1e-9)
                ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)

                if ei > best_ei:
                    best_ei = ei
                    best_candidate = d

            val = self._evaluate(best_candidate, "bayesian")
            X.append(self._dict_to_array(best_candidate)); y.append(val)
            best = np.max(y) if maximize else np.min(y)

        idx = int(np.argmax(y) if maximize else np.argmin(y))
        return self._array_to_dict(X[idx]), float(y[idx])

    def evolutionary_optimize(self, popsize=12, maxiter=40):
        bounds_array = [self.bounds[k] for k in self.param_names]
        def wrap(x): return self._evaluate(self._array_to_dict(x), "evolutionary")
        res = differential_evolution(wrap, bounds_array, popsize=popsize, maxiter=maxiter, seed=42, polish=True, workers=1)
        return self._array_to_dict(res.x), float(res.fun)

    def optimize_ensemble(self):
        p1,v1 = self.bayesian_optimize()
        p2,v2 = self.evolutionary_optimize()
        if v2 < v1:
            return p2, v2, {"bayesian": (p1,v1), "evolutionary": (p2,v2)}
        return p1, v1, {"bayesian": (p1,v1), "evolutionary": (p2,v2)}

# ============================================================
# 11) BUILD A "LONGEVITY PROXY" TARGET (example)
# ============================================================

def build_longevity_proxy(df: pd.DataFrame) -> pd.Series:
    """
    Example proxy: weighted standardized risk from available biomarkers.
    This is NOT a medical truth; it is a modeling target you can replace.
    Lower is "better" (optimization will minimize).
    """
    candidates = []
    # Prefer standardized versions if present
    for col in ["SBP", "DBP", "BMI"]:
        if col in df.columns:
            candidates.append(col)

    # If no candidates, fall back to any numeric columns (avoid collapse)
    if not candidates:
        num = df.select_dtypes(include=[np.number]).columns.tolist()
        candidates = [c for c in num if c != "SEQN"][:8]

    X = df[candidates].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    # z-score
    Z = (X - X.mean()) / (X.std() + 1e-9)
    # weights: equal for demo
    score = Z.mean(axis=1)
    return score.rename("LongevityRiskProxy")

# ============================================================
# 12) MAIN: INGEST -> CLEAN -> VALIDATE -> DISCOVER -> ATTRIB -> SIM -> OPT
# ============================================================

# ---- A) NHANES ingest (choose cycles) ----
NHANES_BEGIN_YEARS = [2017, 2015]  # 2017-18, 2015-16 (adjust)
nhanes_frames = []

for by in NHANES_BEGIN_YEARS:
    xpt_paths = nhanes_download_cycle(by, max_files=35)  # raise if you have disk
    wide = nhanes_merge_on_seqn(xpt_paths, key="SEQN")
    wide["NHANES_BeginYear"] = by
    nhanes_frames.append(wide)

nhanes = pd.concat(nhanes_frames, ignore_index=True, sort=False)
print("NHANES merged shape:", nhanes.shape)

# ---- B) Clean NHANES ----
nhanes = coerce_numeric(nhanes, min_numeric_ratio=0.98)
nhanes = normalize_missing_codes(nhanes)
nhanes = basic_health_feature_pack(nhanes)

# Keep columns with enough signal
min_nonnull = 0.15
keep = [c for c in nhanes.columns if nhanes[c].notna().mean() >= min_nonnull]
nhanes = nhanes[keep].copy()
print("NHANES after keep:", nhanes.shape)

# ---- C) Build target proxy ----
nhanes["LongevityRiskProxy"] = build_longevity_proxy(nhanes)

# ---- D) Validation (cycle-to-cycle drift) ----
ref = nhanes[nhanes["NHANES_BeginYear"] == NHANES_BEGIN_YEARS[0]].sample(
    n=min(5000, (nhanes["NHANES_BeginYear"] == NHANES_BEGIN_YEARS[0]).sum()),
    random_state=42
)
titan = TitanValidationFramework(ref)

for by in NHANES_BEGIN_YEARS[1:]:
    new = nhanes[nhanes["NHANES_BeginYear"] == by].sample(
        n=min(5000, (nhanes["NHANES_BeginYear"] == by).sum()),
        random_state=42
    )
    safe, report = titan.validate(
        new_data=new,
        target_col="LongevityRiskProxy",
        consistency_rules=["BMI > 0"] if "BMI" in new.columns else None
    )
    print("\nTVF verdict for begin year", by, "SAFE?" , safe)
    print(report["traffic_light"].to_string(index=False))

# ---- E) Discovery (proxy kill + regimes) ----
# Define anchors: age/sex (if present) + a small set of stable vitals
anchor_cols = [c for c in ["AgeYears","Sex","BMI","SBP","DBP"] if c in nhanes.columns]
feature_cols = [c for c in nhanes.select_dtypes(include=[np.number]).columns if c not in ["SEQN","LongevityRiskProxy"]]
suspects = [c for c in feature_cols if c not in anchor_cols]

disc_df = nhanes[anchor_cols + suspects + ["LongevityRiskProxy"]].dropna(subset=["LongevityRiskProxy"]).copy()
disc_df = disc_df.fillna(0)

engine = UnifiedDiscoveryEngine(disc_df, target_col="LongevityRiskProxy")
print("Discovery baseline:", engine.scan_baseline())

if anchor_cols and suspects:
    out = engine.proxy_kill_dml(anchor_cols=anchor_cols, suspect_cols=suspects)
    print("Kept features after proxy-kill:", len(out["kept_features"]))

# Regime split example: age decade if AgeYears exists
if "AgeYears" in disc_df.columns:
    disc_df["AgeDecade"] = (disc_df["AgeYears"] // 10).astype(int)
    engine2 = UnifiedDiscoveryEngine(disc_df[engine.known_features + ["AgeDecade","LongevityRiskProxy"]],
                                    target_col="LongevityRiskProxy", group_col="AgeDecade")
    print("Regime scan (AgeDecade):", engine2.detect_regimes(split_col="AgeDecade", min_group_n=400))

# ---- F) GPU Attribution model ----
# Train on a manageable sample for Kaggle
model_cols = [c for c in engine.known_features if c in disc_df.columns]
X = disc_df[model_cols].astype(np.float32)
y = disc_df["LongevityRiskProxy"].astype(np.float32)

# Subsample for speed
N = min(len(X), 40000)
idx = np.random.choice(len(X), N, replace=False)
Xn = X.iloc[idx].copy()
yn = y.iloc[idx].copy()

attrib = UniversalAttributionValidator(Xn, yn, hidden=96, epochs=250)
ig = attrib.integrated_gradients(steps=64)
print("\nTop IG drivers:")
print(ig.head(15).to_string())

inter = attrib.pair_interactions(top_pairs=10)
print("\nTop interaction pairs:")
print(inter.to_string())

# ---- G) Causal simulation (what-if) ----
# Build a *simple* causal graph; revise as you learn more.
# Example structure (edit freely):
# Age -> BMI, SBP, DBP ; BMI -> SBP, DBP ; SBP/DBP/BMI -> LongevityRiskProxy
graph = []
if "AgeYears" in disc_df.columns:
    for v in ["BMI","SBP","DBP"]:
        if v in disc_df.columns:
            graph.append(("AgeYears", v))
if "BMI" in disc_df.columns:
    for v in ["SBP","DBP"]:
        if v in disc_df.columns:
            graph.append(("BMI", v))
for v in ["BMI","SBP","DBP","AgeYears","Sex"]:
    if v in disc_df.columns and v != "LongevityRiskProxy":
        graph.append((v, "LongevityRiskProxy"))

sim_cols = sorted(set([a for a,b in graph] + [b for a,b in graph]))
sim_df = disc_df[sim_cols].copy().fillna(0)

ce = PlatinumCausalEngine(graph, sim_df, verbose=True)

# Example interventions (purely hypothetical; for model exploration)
treatments = []
if "BMI" in sim_df.columns:
    treatments.append({"BMI": float(sim_df["BMI"].median()) - 2.0})
if "SBP" in sim_df.columns:
    treatments.append({"SBP": float(sim_df["SBP"].median()) - 8.0})
if "BMI" in sim_df.columns and "SBP" in sim_df.columns:
    treatments.append({"BMI": float(sim_df["BMI"].median()) - 2.0, "SBP": float(sim_df["SBP"].median()) - 8.0})

print("\nCausal what-if simulations (E[proxy | do(.)]):")
for t in treatments:
    res = ce.simulate_intervention(t, target="LongevityRiskProxy", n_boot=50)
    print(t, res)

# ---- H) Optimization loop (search treatments that minimize risk proxy) ----
bounds = {}
if "BMI" in sim_df.columns:
    bounds["BMI"] = (float(sim_df["BMI"].quantile(0.05)), float(sim_df["BMI"].quantile(0.95)))
if "SBP" in sim_df.columns:
    bounds["SBP"] = (float(sim_df["SBP"].quantile(0.05)), float(sim_df["SBP"].quantile(0.95)))
if "DBP" in sim_df.columns:
    bounds["DBP"] = (float(sim_df["DBP"].quantile(0.05)), float(sim_df["DBP"].quantile(0.95)))

if len(bounds) >= 2:
    def objective(**params):
        # minimize expected risk proxy under do(params)
        r = ce.simulate_intervention(params, target="LongevityRiskProxy", n_boot=0)
        return r["E_y_do"]

    opt = UnifiedOptimizer(objective_fn=objective, bounds=bounds)
    best_params, best_val, all_res = opt.optimize_ensemble()
    print("\nBEST (model-space) intervention found:", best_params)
    print("Expected proxy under do(best):", best_val)
    print("All results:", all_res)
else:
    print("\nNot enough controllable variables found for optimization; expand bounds or feature pack.")

# ---- I) Save artifacts ----
out_csv = os.path.join(BASE_DIR, "nhanes_health_merged.csv")
nhanes.to_csv(out_csv, index=False)
print("\nSaved merged NHANES to:", out_csv)

In [ ]:
# ==============================================================================
# "ETERNAL FOUNDATION" - RAPID GPU HEALTH OPTIMIZATION (v2.0 Fast-Track)
# ==============================================================================
# A streamlined, high-speed implementation of the 5-Framework Architecture
# designed to ingest, validate, dissect, and optimize human biology data
# in under 10 minutes on Kaggle GPU.
# ==============================================================================

import os
import requests
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
import xgboost as xgb
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.model_selection import KFold
from scipy.optimize import differential_evolution
from scipy.stats import qmc
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 SYSTEM ONLINE: {DEVICE} | Mode: RAPID FOUNDATION")

# ==============================================================================
# 1. DATA INGESTION: THE "WHOLE BODY" SNAPSHOT (Hardcoded for Speed)
# ==============================================================================
# We grab 5 specific high-value files from NHANES 2017-2018 (Cycle J).
# No crawling. No waiting.
# ==============================================================================

NHANES_BASE = "https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/"
FILES = {
    "DEMO": "DEMO_J.XPT",    # Age, Sex, Education
    "BMX":  "BMX_J.XPT",     # Body Measures (Weight, BMI, Waist)
    "BPX":  "BPX_J.XPT",     # Blood Pressure
    "BIOPRO": "BIOPRO_J.XPT",# Std Biochemistry (Albumin, Glucose, Iron, etc.)
    "TCHOL": "TCHOL_J.XPT"   # Total Cholesterol
}

def download_and_merge_fast():
    print("\n>> [PHASE 1] DOWNLOADING WHOLE BODY DATA...")
    dfs = []
    
    for name, fname in tqdm(FILES.items(), desc="Downloading Files"):
        url = NHANES_BASE + fname
        local_path = f"/kaggle/working/{fname}"
        
        # Stream download to avoid memory spikes
        if not os.path.exists(local_path):
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
        
        # Read XPT
        df = pd.read_sas(local_path, format='xport')
        # Cleanup columns (bytes to string)
        df.columns = [c.decode('utf-8') if isinstance(c, bytes) else c for c in df.columns]
        
        # Key subsetting to keep RAM light
        if name == "DEMO":
            df = df[['SEQN', 'RIDAGEYR', 'RIAGENDR']]
            df.columns = ['SEQN', 'Age', 'Sex'] # 1=Male, 2=Female
        elif name == "BMX":
            df = df[['SEQN', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST']]
            df.columns = ['SEQN', 'Weight_kg', 'Height_cm', 'BMI', 'Waist_cm']
        elif name == "BPX":
            # Average 3 readings if available
            sy_cols = [c for c in df.columns if 'BPXSY' in c]
            di_cols = [c for c in df.columns if 'BPXDI' in c]
            df['SBP'] = df[sy_cols].mean(axis=1)
            df['DBP'] = df[di_cols].mean(axis=1)
            df = df[['SEQN', 'SBP', 'DBP']]
        elif name == "BIOPRO":
            # Rename cryptic codes to biological names
            # LBXSAL=Albumin, LBXSGL=Glucose, LBXSTP=TotalProtein, LBXSCA=Calcium
            # LBXSKSI=Potassium, LBXSNASI=Sodium, LBXSCR=Creatinine
            keep = ['SEQN', 'LBXSAL', 'LBXSGL', 'LBXSTP', 'LBXSCA', 'LBXSKSI', 'LBXSNASI', 'LBXSCR']
            df = df[[c for c in keep if c in df.columns]]
            df = df.rename(columns={
                'LBXSAL': 'Albumin', 'LBXSGL': 'Glucose', 'LBXSTP': 'Protein_Total',
                'LBXSCA': 'Calcium', 'LBXSKSI': 'Potassium', 'LBXSNASI': 'Sodium',
                'LBXSCR': 'Creatinine'
            })
        elif name == "TCHOL":
            df = df[['SEQN', 'LBXTC']]
            df.columns = ['SEQN', 'Cholesterol_Total']
            
        dfs.append(df)

    # Merge all on SEQN
    print(">> Merging 'Whole Body' Dataset...")
    master = dfs[0]
    for d in dfs[1:]:
        master = master.merge(d, on='SEQN', how='inner')
    
    # Drop rows with missing criticals
    master = master.dropna()
    print(f"   ✓ DATA READY. Shape: {master.shape} (Humans x Biomarkers)")
    return master

# ==============================================================================
# 2. THE 5 FRAMEWORKS (Optimized "Lite" Classes)
# ==============================================================================

# --- FRAMEWORK 1: TITAN VALIDATION (Lite) ---
class TitanValidationLite:
    def __init__(self, df):
        self.ref = df.copy()
        self.scaler = StandardScaler()
        # Only fit on numeric
        self.num = self.ref.select_dtypes(include=np.number).columns.tolist()
        self.forest = IsolationForest(contamination=0.01, n_jobs=-1, random_state=42)
        self.forest.fit(self.ref[self.num])
        
    def check_integrity(self, df_new):
        # Fast integrity check
        issues = []
        # 1. Structure Check
        if len(df_new.columns) != len(self.ref.columns): issues.append("Column Mismatch")
        # 2. Anomaly Scan
        preds = self.forest.predict(df_new[self.num])
        anomaly_rate = (preds == -1).mean()
        if anomaly_rate > 0.05: issues.append(f"High Anomaly Rate ({anomaly_rate:.1%})")
        return issues if issues else ["✅ INTEGRITY STABLE"]

# --- FRAMEWORK 2: DISCOVERY ENGINE (Lite) ---
class DiscoveryEngineLite:
    def __init__(self, df, target):
        self.df = df
        self.target = target
        
    def find_drivers(self):
        # Quick Random Forest feature importance
        X = self.df.drop(columns=[self.target, 'SEQN'])
        y = self.df[self.target]
        m = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
        m.fit(X, y)
        imp = pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False)
        return imp

# --- FRAMEWORK 3: ATTRIBUTION VALIDATOR (GPU) ---
class AttributionValidatorGPU:
    def __init__(self, X, y):
        self.features = X.columns.tolist()
        self.scaler = StandardScaler()
        X_s = self.scaler.fit_transform(X)
        
        self.X_t = torch.tensor(X_s, dtype=torch.float32).to(DEVICE)
        self.y_t = torch.tensor(y.values, dtype=torch.float32).reshape(-1,1).to(DEVICE)
        
        # Fast MLP
        self.model = nn.Sequential(
            nn.Linear(len(self.features), 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        ).to(DEVICE)
        
        opt = torch.optim.Adam(self.model.parameters(), lr=0.01)
        loss_fn = nn.MSELoss()
        
        # Quick train (50 epochs is enough for foundational signal)
        self.model.train()
        for _ in range(50):
            opt.zero_grad()
            out = self.model(self.X_t)
            loss = loss_fn(out, self.y_t)
            loss.backward()
            opt.step()
        self.model.eval()
        
    def explain(self):
        # Integrated Gradients (Simplified/Fast)
        baseline = torch.zeros_like(self.X_t)
        steps = 20 # reduced from 100 for speed
        alphas = torch.linspace(0, 1, steps).to(DEVICE)
        
        # Batch processing not needed for small N, but good practice
        # We do average gradient over path
        grads_accum = torch.zeros_like(self.X_t)
        
        for a in alphas:
            x_step = baseline + a * (self.X_t - baseline)
            x_step.requires_grad = True
            out = self.model(x_step)
            grad = torch.autograd.grad(out.sum(), x_step)[0]
            grads_accum += grad
            
        avg_grad = grads_accum / steps
        attr = (self.X_t - baseline) * avg_grad
        return pd.Series(attr.mean(dim=0).detach().cpu().numpy(), index=self.features)

# --- FRAMEWORK 4: INTERVENTION ENGINE (Causal Simulator) ---
class InterventionEngineLite:
    def __init__(self, df, graph_edges):
        self.df = df
        self.G = nx.DiGraph(graph_edges)
        self.models = {}
        self.residuals = {}
        self.order = list(nx.topological_sort(self.G))
        
        # Fit Structural Equations (Ridge Regression for speed/stability)
        for node in self.order:
            parents = list(self.G.predecessors(node))
            if parents:
                m = Ridge(alpha=1.0)
                m.fit(self.df[parents], self.df[node])
                self.models[node] = m
                self.residuals[node] = self.df[node] - m.predict(self.df[parents])
            else:
                self.residuals[node] = self.df[node]
                
    def simulate(self, intervention_dict, target):
        # Monte Carlo Simulation
        N = 1000 # Sim population
        sim_df = pd.DataFrame(index=range(N))
        
        # Bootstrap residuals
        res_sample = {n: self.residuals[n].sample(N, replace=True).values for n in self.order}
        
        for node in self.order:
            if node in intervention_dict:
                sim_df[node] = intervention_dict[node]
            else:
                parents = list(self.G.predecessors(node))
                if not parents:
                    sim_df[node] = res_sample[node]
                else:
                    pred = self.models[node].predict(sim_df[parents])
                    sim_df[node] = pred + res_sample[node]
                    
        return sim_df[target].mean()

# --- FRAMEWORK 5: OPTIMIZER (Evolutionary) ---
def optimize_human_body(engine, target_var, bounds):
    print("\n>> [PHASE 5] OPTIMIZING HUMAN CASUAL ELEMENTS...")
    
    def objective(x):
        # Unpack vector to dict
        intv = {k: v for k, v in zip(bounds.keys(), x)}
        # We want to MAXIMIZE score, so minimize negative
        return -engine.simulate(intv, target_var)
    
    limits = list(bounds.values())
    
    # Differential Evolution: Robust global search
    res = differential_evolution(
        objective, 
        limits, 
        maxiter=10,  # Fast iterations
        popsize=10, 
        polish=False,
        disp=True
    )
    
    return {k: v for k, v in zip(bounds.keys(), res.x)}, -res.fun

# ==============================================================================
# 3. EXECUTION PIPELINE
# ==============================================================================

def run_foundation_pipeline():
    # 1. GET DATA
    df = download_and_merge_fast()
    
    # 2. CREATE "LONGEVITY PROXY"
    # We define "Longevity" as high physiological stability.
    # We construct a composite score: Metabolic Integrity.
    # Higher is better.
    # Logic: Lower BMI (to a point), Lower BP, Glucose in range, Albumin high (liver health).
    # We simply use a negative summation of Z-scores of risk factors + positive Z-scores of protective factors.
    
    print("\n>> [PHASE 2] CONSTRUCTING LONGEVITY TARGET...")
    scaler = StandardScaler()
    z = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    
    # Formula: Albumin (Good) - Abs(BMI deviation) - SBP - Glucose - Cholesterol
    # Note: This is a simplistic proxy for the demo.
    df['LongevityScore'] = (
        z['Albumin'] 
        - z['BMI'].abs() 
        - z['SBP'] 
        - z['Glucose'] 
        - z['Cholesterol_Total']
    )
    
    # 3. VALIDATE
    print("\n>> [PHASE 3] TITAN VALIDATION SCAN...")
    tv = TitanValidationLite(df)
    # Validate a random split (Train vs Test)
    test_split = df.sample(frac=0.2)
    print("   Status:", tv.check_integrity(test_split))
    
    # 4. DISCOVER & ATTRIBUTE
    print("\n>> [PHASE 4] GPU DISCOVERY & ATTRIBUTION...")
    # Find what actually moves the needle on 'LongevityScore'
    target = 'LongevityScore'
    features = [c for c in df.columns if c not in [target, 'SEQN']]
    
    # Discovery
    de = DiscoveryEngineLite(df, target)
    drivers = de.find_drivers()
    print("   Top Random Forest Drivers:")
    print(drivers.head(5).to_string())
    
    # Attribution (GPU)
    av = AttributionValidatorGPU(df[features], df[target])
    ig_scores = av.explain()
    print("\n   Top Neural Attribution (IG) Drivers:")
    print(ig_scores.sort_values(ascending=False).head(5).to_string())
    
    # 5. CAUSAL OPTIMIZATION
    # Build Graph: Age/Sex -> BMI/Waist -> BP/Glu/Chol/Alb -> Longevity
    # We want to find the optimal BMI, Glucose, etc. for a given Age/Sex.
    graph = [
        ('Age', 'BMI'), ('Sex', 'BMI'),
        ('Age', 'Waist_cm'), ('Sex', 'Waist_cm'),
        ('BMI', 'SBP'), ('BMI', 'Glucose'), ('BMI', 'Cholesterol_Total'),
        ('Waist_cm', 'SBP'), ('Waist_cm', 'Glucose'),
        ('SBP', 'LongevityScore'),
        ('Glucose', 'LongevityScore'),
        ('Cholesterol_Total', 'LongevityScore'),
        ('Albumin', 'LongevityScore'),
        ('Waist_cm', 'LongevityScore'), # Direct link
        ('BMI', 'LongevityScore')       # Direct link
    ]
    
    ie = InterventionEngineLite(df, graph)
    
    # Search Space: What can we "edit"? 
    # We can't edit Age/Sex. We CAN edit Weight, Diet (via Glucose/Chol), BP (via meds/exercise).
    # Let's optimize: BMI, SBP, Glucose, Cholesterol.
    bounds = {
        'BMI': (18.5, 30.0),          # Healthy range search
        'SBP': (110.0, 140.0),        # BP range
        'Glucose': (70.0, 120.0),     # Sugar range
        'Cholesterol_Total': (130.0, 240.0)
    }
    
    best_params, best_score = optimize_human_body(ie, 'LongevityScore', bounds)
    
    print("\n" + "="*60)
    print("🌟 FINAL OPTIMIZATION RESULTS (FOUNDATIONAL RUN) 🌟")
    print("="*60)
    print(f"Goal: Maximize Metabolic Integrity (Longevity Proxy)")
    print(f"Base Average Score: {df['LongevityScore'].mean():.4f}")
    print(f"OPTIMIZED Score:    {best_score:.4f}")
    print("-" * 30)
    print("OPTIMAL BODY PARAMETERS (The 'Blueprint'):")
    for k, v in best_params.items():
        print(f"  • {k:20s}: {v:.2f}")
    print("="*60)
    print(f"Completed successfully on {DEVICE}.")

if __name__ == "__main__":
    run_foundation_pipeline()